# Patch-level PCA for MID and JNG

`processed/*_clean.csv` 파일을 다시 불러와 선수-포지션-패치 단위로 집계한 뒤 PCA를 수행합니다.

- 전체 경기력 PCA: 전반적인 개인 경기력 지표 사용
- 라인전 PCA: 15분 골드, 경험치, CS 지표만 사용
- `16.0`은 시즌 16 범위로 해석하여 현재 파일에 존재하는 `16.xx` 패치를 포함합니다.
- 미드와 정글의 지표 분포 차이를 고려하여 기본값은 포지션별 PCA입니다.

## 1. 라이브러리와 설정

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("processed")
OUTPUT_DIR = PROCESSED_DIR / "pca_patch_mid_jng"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POSITIONS = ["mid", "jng"]
PATCH_MIN = (14, 1)       # 14.01
PATCH_MAX = (16, 99)      # 시즌 16의 현재 보유 패치까지 포함
MIN_MATCHES_PER_PATCH = 3 # 선수-포지션-패치별 최소 경기 수
PCA_BY_POSITION = True    # mid와 jng를 각각 PCA

print("output:", OUTPUT_DIR.resolve())

## 2. 전처리 파일 불러오기와 패치 필터링

In [ ]:
def parse_patch(value):
    """'14.01', '15.1', 16.01 같은 값을 비교 가능한 (major, minor) 튜플로 변환합니다."""
    match = re.match(r"^\s*(\d+)\.(\d+)\s*$", str(value))
    if not match:
        return None
    return int(match.group(1)), int(match.group(2))


def format_patch(patch_tuple):
    return f"{patch_tuple[0]}.{patch_tuple[1]:02d}"


clean_files = sorted(PROCESSED_DIR.glob("*_clean.csv"))
if not clean_files:
    raise FileNotFoundError("processed/*_clean.csv 파일을 찾지 못했습니다.")

frames = []
for file in clean_files:
    df = pd.read_csv(file)
    required = {"playername", "position", "patch"}
    if required.issubset(df.columns):
        df["source_file"] = file.name
        frames.append(df)

matches = pd.concat(frames, ignore_index=True)
matches["patch_tuple"] = matches["patch"].map(parse_patch)
matches = matches[matches["patch_tuple"].notna()].copy()
matches = matches[matches["position"].isin(POSITIONS)].copy()
matches = matches[matches["patch_tuple"].map(lambda x: PATCH_MIN <= x <= PATCH_MAX)].copy()
if matches.empty:
    raise ValueError("설정한 패치 범위와 포지션에 해당하는 데이터가 없습니다.")
matches["patch"] = matches["patch_tuple"].map(format_patch)
matches["patch_major"] = matches["patch_tuple"].map(lambda x: x[0])
matches["patch_minor"] = matches["patch_tuple"].map(lambda x: x[1])

print("사용 파일:", sorted(matches["source_file"].unique()))
print("경기 row 수:", len(matches))
print("패치 범위:", matches["patch"].iloc[0], "~", matches["patch"].iloc[-1])
display(matches.groupby(["patch_major", "patch_minor", "patch", "position"]).size().rename("rows").reset_index())

## 3. PCA용 지표 생성과 선수-패치 단위 집계

경기 길이 영향을 줄이기 위해 일부 raw count 대신 분당 지표를 추가합니다. PCA 입력 지표는 아래 목록에서 관리합니다.

In [ ]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


numeric_candidates = matches.columns.difference(["playername", "teamname", "position", "patch", "date", "league", "datacompleteness", "source_file", "patch_tuple"])
matches[numeric_candidates] = matches[numeric_candidates].apply(pd.to_numeric, errors="coerce")

minutes = matches["gamelength"] / 60
matches["kill_participation"] = safe_divide(matches["kills"] + matches["assists"], matches["teamkills"])
matches["deaths_per_min"] = safe_divide(matches["deaths"], minutes)
matches["tower_damage_per_min"] = safe_divide(matches["damagetotowers"], minutes)
matches["controlwards_per_min"] = safe_divide(matches["controlwardsbought"], minutes)

LANE_FEATURES = [
    "golddiffat15", "xpdiffat15", "csdiffat15",
    "goldat15", "xpat15", "csat15",
]

FULL_FEATURES = [
    "golddiffat15", "xpdiffat15", "csdiffat15",
    "kills", "assists", "deaths_per_min", "kill_participation",
    "dpm", "damageshare", "damagetakenperminute", "damagemitigatedperminute",
    "visionscore", "vspm", "wpm", "wcpm", "controlwards_per_min",
    "earned gpm", "earnedgoldshare", "cspm", "tower_damage_per_min",
    "firstbloodkill", "firstbloodassist", "firstbloodvictim",
]

FULL_FEATURES = [column for column in FULL_FEATURES if column in matches.columns]
LANE_FEATURES = [column for column in LANE_FEATURES if column in matches.columns]

group_keys = ["playername", "position", "patch", "patch_major", "patch_minor"]
agg_spec = {feature: "mean" for feature in sorted(set(FULL_FEATURES + LANE_FEATURES))}
agg_spec["teamname"] = lambda values: " | ".join(sorted(set(values.dropna().astype(str))))

player_patch = matches.groupby(group_keys, as_index=False).agg(agg_spec)
match_counts = matches.groupby(group_keys).size().rename("match_count").reset_index()
player_patch = player_patch.merge(match_counts, on=group_keys, how="left")
player_patch["low_sample_flag"] = (player_patch["match_count"] < MIN_MATCHES_PER_PATCH).astype(int)
player_patch = player_patch.sort_values(["patch_major", "patch_minor", "position", "playername"]).reset_index(drop=True)

player_patch.to_csv(OUTPUT_DIR / "player_patch_features.csv", index=False, encoding="utf-8-sig")
print("선수-패치 row 수:", len(player_patch))
print("최소 경기 수 충족 row 수:", (player_patch["low_sample_flag"] == 0).sum())
display(player_patch.head())

## 4. PCA 공통 함수

In [ ]:
ID_COLUMNS = ["playername", "teamname", "position", "patch", "patch_major", "patch_minor", "match_count", "low_sample_flag"]


def run_pca(data, feature_columns, analysis_name, by_position=PCA_BY_POSITION):
    eligible = data[data["low_sample_flag"] == 0].copy()
    groups = eligible.groupby("position", sort=True) if by_position else [("mid_jng", eligible)]
    score_frames, loading_frames, variance_frames = [], [], []

    for group_name, group in groups:
        usable = [column for column in feature_columns if group[column].notna().any() and group[column].nunique(dropna=True) > 1]
        if len(group) < 2 or len(usable) < 2:
            print(f"skip: {analysis_name} / {group_name} (rows={len(group)}, features={len(usable)})")
            continue

        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        matrix = scaler.fit_transform(imputer.fit_transform(group[usable]))
        pca = PCA(n_components=min(matrix.shape))
        transformed = pca.fit_transform(matrix)
        pc_names = [f"PC{i}" for i in range(1, transformed.shape[1] + 1)]

        scores = group[ID_COLUMNS].reset_index(drop=True).copy()
        scores["pca_group"] = group_name
        scores = pd.concat([scores, pd.DataFrame(transformed, columns=pc_names)], axis=1)

        loadings = pd.DataFrame(pca.components_.T, index=usable, columns=pc_names).reset_index(names="feature")
        loadings.insert(0, "pca_group", group_name)

        variance = pd.DataFrame({
            "pca_group": group_name,
            "component": pc_names,
            "explained_variance_ratio": pca.explained_variance_ratio_,
            "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
        })

        score_frames.append(scores)
        loading_frames.append(loadings)
        variance_frames.append(variance)

    scores = pd.concat(score_frames, ignore_index=True)
    loadings = pd.concat(loading_frames, ignore_index=True)
    variance = pd.concat(variance_frames, ignore_index=True)

    scores.to_csv(OUTPUT_DIR / f"{analysis_name}_scores.csv", index=False, encoding="utf-8-sig")
    loadings.to_csv(OUTPUT_DIR / f"{analysis_name}_loadings.csv", index=False, encoding="utf-8-sig")
    variance.to_csv(OUTPUT_DIR / f"{analysis_name}_explained_variance.csv", index=False, encoding="utf-8-sig")
    return scores, loadings, variance


def plot_variance(variance, title):
    groups = variance["pca_group"].unique()
    fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 4), squeeze=False)
    for ax, group_name in zip(axes[0], groups):
        current = variance[variance["pca_group"] == group_name]
        x = np.arange(1, len(current) + 1)
        ax.bar(x, current["explained_variance_ratio"], alpha=0.7, label="individual")
        ax.plot(x, current["cumulative_explained_variance_ratio"], marker="o", label="cumulative")
        ax.axhline(0.8, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{title}: {group_name}")
        ax.set_xlabel("Principal component")
        ax.set_ylabel("Explained variance ratio")
        ax.legend()
    plt.tight_layout()
    plt.show()

## 5. 전체 경기력 PCA

In [ ]:
full_scores, full_loadings, full_variance = run_pca(
    player_patch,
    FULL_FEATURES,
    analysis_name="full_performance_pca",
)

plot_variance(full_variance, "Full performance PCA")
display(full_variance.groupby("pca_group").head(5))
display(full_loadings[["pca_group", "feature", "PC1"]].assign(abs_PC1=lambda x: x["PC1"].abs()).sort_values(["pca_group", "abs_PC1"], ascending=[True, False]).groupby("pca_group").head(10))

## 6. 라인전 PCA

In [ ]:
lane_scores, lane_loadings, lane_variance = run_pca(
    player_patch,
    LANE_FEATURES,
    analysis_name="laning_pca",
)

plot_variance(lane_variance, "Laning PCA")
display(lane_variance.groupby("pca_group").head(5))
display(lane_loadings[["pca_group", "feature", "PC1"]].assign(abs_PC1=lambda x: x["PC1"].abs()).sort_values(["pca_group", "abs_PC1"], ascending=[True, False]))

## 7. 저장 결과 확인

In [ ]:
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(file)